# Multi-Query Attention (MQA) and Grouped-Query Attention (GQA)

**Stage 2: Memory Optimization - Notebook 21**

This notebook explores MQA and GQA, techniques that dramatically reduce KV cache size. We'll cover:
- Standard multi-head attention (MHA) review
- The KV cache memory problem
- Multi-Query Attention (MQA) - share K,V across heads
- Grouped-Query Attention (GQA) - balance quality and efficiency
- KV cache size calculations and comparisons
- Implementation from scratch
- Quality vs efficiency trade-offs
- When to use MQA vs GQA
- Models using each approach (Llama 2 70B, Mistral, etc.)

**References**:
- LLM_INFERENCE_ROADMAP.md - Stage 2
- Notebook 20: Flash Attention Explained
- Notebook 55: KV Cache Optimization
- Paper: "Fast Transformer Decoding: One Write-Head is All You Need" (Shazeer, 2019)
- Paper: "GQA: Training Generalized Multi-Query Transformer" (Ainslie et al., 2023)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
import math
from dataclasses import dataclass

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Standard Multi-Head Attention (MHA) Review

### The KV Cache Memory Problem

**Standard Multi-Head Attention**:
```python
# Each head has its own Q, K, V projections
for head in range(num_heads):
    Q_head = Q[:, head]  # [batch, seq, head_dim]
    K_head = K[:, head]  # [batch, seq, head_dim]
    V_head = V[:, head]  # [batch, seq, head_dim]
    
    attn = softmax(Q_head @ K_head.T / sqrt(d_k))
    output_head = attn @ V_head
```

**KV Cache Size (Standard MHA)**:
```
Per layer: 2 × seq_len × num_heads × head_dim × dtype_bytes
All layers: 2 × seq_len × num_heads × head_dim × num_layers × dtype_bytes

Example (Llama 2 70B, 4K context):
- seq_len: 4096
- num_heads: 64
- head_dim: 128
- num_layers: 80
- dtype: FP16 (2 bytes)

KV cache = 2 × 4096 × 64 × 128 × 80 × 2 bytes
          = 10.7 GB per sequence!
          
With batch_size=32: 342 GB just for KV cache!
```

**Problem**: KV cache dominates memory usage for long sequences and large batches.

In [ ]:
def calculate_kv_cache_size(seq_len, num_heads, head_dim, num_layers, 
                            batch_size=1, dtype_bytes=2, num_kv_heads=None):
    """
    Calculate KV cache memory requirements.
    
    Args:
        seq_len: Sequence length
        num_heads: Number of query heads
        head_dim: Dimension per head
        num_layers: Number of transformer layers
        batch_size: Batch size
        dtype_bytes: Bytes per element (2 for FP16)
        num_kv_heads: Number of KV heads (for GQA). If None, equals num_heads (MHA)
    
    Returns:
        Dictionary with memory breakdown
    """
    if num_kv_heads is None:
        num_kv_heads = num_heads  # Standard MHA
    
    # KV cache: 2 (K and V) × batch × seq × num_kv_heads × head_dim × layers
    kv_cache_bytes = 2 * batch_size * seq_len * num_kv_heads * head_dim * num_layers * dtype_bytes
    
    return {
        'total_gb': kv_cache_bytes / 1e9,
        'per_layer_mb': (kv_cache_bytes / num_layers) / 1e6,
        'per_token_kb': (kv_cache_bytes / (batch_size * seq_len)) / 1e3,
    }


print("KV Cache Memory Analysis")
print("=" * 100)

# Common model configurations
configs = [
    ('Llama 2 7B', 4096, 32, 128, 32, 1),
    ('Llama 2 13B', 4096, 40, 128, 40, 1),
    ('Llama 2 70B', 4096, 64, 128, 80, 1),
    ('Llama 2 70B (batch=32)', 4096, 64, 128, 80, 32),
    ('Llama 2 70B (8K ctx)', 8192, 64, 128, 80, 1),
    ('Llama 2 70B (32K ctx)', 32768, 64, 128, 80, 1),
]

print(f"{'Model':<30} {'Seq Len':<10} {'Batch':<8} {'KV Cache (GB)':<15} {'Per Layer (MB)'}")
print("-" * 100)

for name, seq_len, num_heads, head_dim, num_layers, batch_size in configs:
    result = calculate_kv_cache_size(seq_len, num_heads, head_dim, num_layers, batch_size)
    print(f"{name:<30} {seq_len:<10} {batch_size:<8} {result['total_gb']:<15.2f} {result['per_layer_mb']:.1f}")

print("\n" + "=" * 100)
print("\nKey Observations:")
print("1. KV cache scales linearly with sequence length")
print("2. KV cache scales linearly with batch size")
print("3. Long contexts (32K) require 100+ GB for single sequence!")
print("4. Large batches quickly exhaust GPU memory")
print("5. Need: Reduce num_kv_heads to reduce KV cache size")

In [ ]:
# Visualize KV cache scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KV cache vs sequence length
ax = axes[0]
seq_lengths = [1024, 2048, 4096, 8192, 16384, 32768]
kv_cache_sizes = []

for seq_len in seq_lengths:
    result = calculate_kv_cache_size(seq_len, num_heads=64, head_dim=128, 
                                     num_layers=80, batch_size=1)
    kv_cache_sizes.append(result['total_gb'])

ax.plot(seq_lengths, kv_cache_sizes, 'o-', linewidth=2, markersize=8, color='#e74c3c')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('KV Cache Size (GB)', fontsize=12)
ax.set_title('KV Cache vs Sequence Length (Llama 2 70B)', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# Add annotations
for x, y in zip(seq_lengths, kv_cache_sizes):
    ax.annotate(f'{y:.1f} GB', xy=(x, y), xytext=(5, 5), 
                textcoords='offset points', fontsize=9)

# KV cache vs batch size
ax = axes[1]
batch_sizes = [1, 2, 4, 8, 16, 32, 64]
kv_cache_batched = []

for batch in batch_sizes:
    result = calculate_kv_cache_size(seq_len=4096, num_heads=64, head_dim=128,
                                     num_layers=80, batch_size=batch)
    kv_cache_batched.append(result['total_gb'])

ax.bar(range(len(batch_sizes)), kv_cache_batched, color='#3498db', alpha=0.7)
ax.set_xticks(range(len(batch_sizes)))
ax.set_xticklabels(batch_sizes)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('KV Cache Size (GB)', fontsize=12)
ax.set_title('KV Cache vs Batch Size (Llama 2 70B, 4K ctx)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add A100 memory limit
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, alpha=0.5, label='A100 80GB')
ax.legend()

plt.tight_layout()
plt.show()

print("\nConclusion: KV cache is a major bottleneck for:")
print("- Long context inference (8K+ tokens)")
print("- Large batch sizes")
print("- Multi-user serving")
print("\nSolution: Multi-Query Attention (MQA) and Grouped-Query Attention (GQA)")

## 2. Multi-Query Attention (MQA)

### The Breakthrough Idea

**Key Insight** (Shazeer, 2019):
```
Q (queries) must be per-head: different heads attend to different patterns
K, V (keys, values) can be shared across all heads!
```

**Standard MHA**:
```python
num_heads = 32
Q: [batch, seq, num_heads, head_dim]  # 32 separate Q matrices
K: [batch, seq, num_heads, head_dim]  # 32 separate K matrices
V: [batch, seq, num_heads, head_dim]  # 32 separate V matrices
```

**Multi-Query Attention (MQA)**:
```python
num_heads = 32
Q: [batch, seq, num_heads, head_dim]  # 32 separate Q matrices
K: [batch, seq, 1, head_dim]          # SHARED across all heads!
V: [batch, seq, 1, head_dim]          # SHARED across all heads!
```

**Memory Reduction**:
```
MHA KV cache: num_heads × head_dim per token
MQA KV cache: 1 × head_dim per token
Reduction: num_heads × smaller (8-32x for typical models!)
```

**Quality Trade-off**:
- Less expressiveness (shared K,V)
- Typical degradation: 1-2% on benchmarks
- Often acceptable for the massive memory savings

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Standard Multi-Head Attention (MHA).
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Separate projections for each head
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None, use_kv_cache=False, past_kv=None):
        batch_size, seq_len, _ = x.shape
        
        # Project Q, K, V
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # Transpose for attention: [batch, num_heads, seq, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # KV cache
        if use_kv_cache and past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        output = torch.matmul(attn, V)
        
        # Reshape and project
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, self.d_model)
        output = self.out_proj(output)
        
        # Return KV cache for next iteration
        kv_cache = (K, V) if use_kv_cache else None
        
        return output, kv_cache


class MultiQueryAttention(nn.Module):
    """
    Multi-Query Attention (MQA).
    
    Key difference: K and V are shared across all heads.
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Q projection: per-head (same as MHA)
        self.q_proj = nn.Linear(d_model, d_model)
        
        # K, V projections: SHARED across heads (single head_dim)
        self.k_proj = nn.Linear(d_model, self.head_dim)
        self.v_proj = nn.Linear(d_model, self.head_dim)
        
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None, use_kv_cache=False, past_kv=None):
        batch_size, seq_len, _ = x.shape
        
        # Project Q: [batch, seq, num_heads, head_dim]
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        Q = Q.transpose(1, 2)  # [batch, num_heads, seq, head_dim]
        
        # Project K, V: [batch, seq, head_dim] (SHARED!)
        K = self.k_proj(x)  # [batch, seq, head_dim]
        V = self.v_proj(x)  # [batch, seq, head_dim]
        
        # KV cache
        if use_kv_cache and past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=1)
            V = torch.cat([past_v, V], dim=1)
        
        # Expand K, V for all heads: [batch, num_heads, seq, head_dim]
        K = K.unsqueeze(1).expand(-1, self.num_heads, -1, -1)
        V = V.unsqueeze(1).expand(-1, self.num_heads, -1, -1)
        
        # Attention (same as MHA)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        output = torch.matmul(attn, V)
        
        # Reshape and project
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, self.d_model)
        output = self.out_proj(output)
        
        # Return KV cache (note: K, V are not expanded in cache!)
        kv_cache = (K[:, 0], V[:, 0]) if use_kv_cache else None  # Only store one copy
        
        return output, kv_cache


print("Multi-Query Attention (MQA) Implementation")
print("=" * 70)
print("\nKey Differences from MHA:")
print("1. Q projection: d_model → d_model (same as MHA)")
print("2. K projection: d_model → head_dim (REDUCED!)")
print("3. V projection: d_model → head_dim (REDUCED!)")
print("4. K, V are expanded/broadcast to all heads during attention")
print("5. KV cache: Only store single K, V (not per-head)")
print("\nResult: num_heads × smaller KV cache!")

In [ ]:
# Compare MHA vs MQA memory
print("\nMHA vs MQA Memory Comparison")
print("=" * 90)

configs = [
    ('Llama 2 7B', 4096, 32, 128, 32),
    ('Llama 2 13B', 4096, 40, 128, 40),
    ('Llama 2 70B', 4096, 64, 128, 80),
]

print(f"{'Model':<15} {'MHA KV Cache (GB)':<20} {'MQA KV Cache (GB)':<20} {'Reduction Factor'}")
print("-" * 90)

for name, seq_len, num_heads, head_dim, num_layers in configs:
    # MHA: full num_heads
    mha_result = calculate_kv_cache_size(seq_len, num_heads, head_dim, num_layers)
    
    # MQA: only 1 KV head
    mqa_result = calculate_kv_cache_size(seq_len, num_heads, head_dim, num_layers, 
                                         num_kv_heads=1)
    
    reduction = mha_result['total_gb'] / mqa_result['total_gb']
    
    print(f"{name:<15} {mha_result['total_gb']:<20.2f} {mqa_result['total_gb']:<20.2f} {reduction:.1f}x")

print("\n" + "=" * 90)
print("\nMQA achieves ~num_heads× reduction in KV cache size!")
print("For 32-64 heads: 32-64x smaller KV cache")

In [ ]:
# Test MHA vs MQA
torch.manual_seed(42)

d_model = 512
num_heads = 8
seq_len = 64
batch_size = 2

x = torch.randn(batch_size, seq_len, d_model, device=device)

# MHA
mha = MultiHeadAttention(d_model, num_heads).to(device)
mha_output, mha_kv = mha(x, use_kv_cache=True)

# MQA
mqa = MultiQueryAttention(d_model, num_heads).to(device)
mqa_output, mqa_kv = mqa(x, use_kv_cache=True)

print("MHA vs MQA Output Shapes")
print("=" * 70)
print(f"Input shape: {x.shape}")
print(f"\nMHA:")
print(f"  Output shape: {mha_output.shape}")
print(f"  KV cache K shape: {mha_kv[0].shape}")
print(f"  KV cache V shape: {mha_kv[1].shape}")
print(f"\nMQA:")
print(f"  Output shape: {mqa_output.shape}")
print(f"  KV cache K shape: {mqa_kv[0].shape}")
print(f"  KV cache V shape: {mqa_kv[1].shape}")

# Calculate memory savings
mha_kv_size = mha_kv[0].numel() + mha_kv[1].numel()
mqa_kv_size = mqa_kv[0].numel() + mqa_kv[1].numel()
reduction = mha_kv_size / mqa_kv_size

print(f"\nKV Cache Memory:")
print(f"  MHA: {mha_kv_size:,} elements")
print(f"  MQA: {mqa_kv_size:,} elements")
print(f"  Reduction: {reduction:.1f}x smaller")

## 3. Grouped-Query Attention (GQA)

### The Best of Both Worlds

**Problem with MQA**:
- Maximum memory savings (num_heads× reduction)
- But: Quality degradation (1-2% on benchmarks)
- Reason: Single K,V for all heads is limiting

**Grouped-Query Attention (GQA)** (Ainslie et al., 2023):
```
Idea: Group heads, share K,V within groups

MHA: num_heads = 32, num_kv_heads = 32 (no sharing)
GQA: num_heads = 32, num_kv_heads = 4  (groups of 8)
MQA: num_heads = 32, num_kv_heads = 1  (full sharing)
```

**Example** (32 query heads):
```python
# GQA with 4 KV heads (groups of 8)
Group 1: Q_heads[0:8]   → shared K_1, V_1
Group 2: Q_heads[8:16]  → shared K_2, V_2
Group 3: Q_heads[16:24] → shared K_3, V_3
Group 4: Q_heads[24:32] → shared K_4, V_4
```

**Trade-offs**:
```
MHA: Best quality, largest KV cache
GQA: Good quality, medium KV cache (sweet spot!)
MQA: Acceptable quality, smallest KV cache
```

**Real-world Usage**:
- Llama 2 70B: GQA with 8 KV heads (64 query heads → 8 groups)
- Mistral 7B: GQA with 8 KV heads (32 query heads → 4 groups)
- GPT-4 (rumored): MQA or GQA
- Falcon: MQA (1 KV head)

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA).
    
    Intermediate between MHA and MQA:
    - Group query heads
    - Share K, V within each group
    """
    def __init__(self, d_model, num_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.num_queries_per_kv = num_heads // num_kv_heads
        
        # Q projection: per-head
        self.q_proj = nn.Linear(d_model, d_model)
        
        # K, V projections: per-group
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)
        
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None, use_kv_cache=False, past_kv=None):
        batch_size, seq_len, _ = x.shape
        
        # Project Q: [batch, seq, num_heads, head_dim]
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        Q = Q.transpose(1, 2)  # [batch, num_heads, seq, head_dim]
        
        # Project K, V: [batch, seq, num_kv_heads, head_dim]
        K = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        
        K = K.transpose(1, 2)  # [batch, num_kv_heads, seq, head_dim]
        V = V.transpose(1, 2)  # [batch, num_kv_heads, seq, head_dim]
        
        # KV cache
        if use_kv_cache and past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        
        # Expand K, V to match number of query heads
        # Repeat each KV head for its group of query heads
        K = K.repeat_interleave(self.num_queries_per_kv, dim=1)
        V = V.repeat_interleave(self.num_queries_per_kv, dim=1)
        # Now K, V are [batch, num_heads, seq, head_dim]
        
        # Attention (same as MHA)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        output = torch.matmul(attn, V)
        
        # Reshape and project
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, self.d_model)
        output = self.out_proj(output)
        
        # Return KV cache (store only num_kv_heads, not expanded)
        if use_kv_cache:
            # Extract original KV heads (before expansion)
            K_cache = K[:, ::self.num_queries_per_kv]
            V_cache = V[:, ::self.num_queries_per_kv]
            kv_cache = (K_cache, V_cache)
        else:
            kv_cache = None
        
        return output, kv_cache


print("Grouped-Query Attention (GQA) Implementation")
print("=" * 70)
print("\nKey Properties:")
print("1. Q projection: d_model → d_model (same as MHA)")
print("2. K projection: d_model → num_kv_heads × head_dim")
print("3. V projection: d_model → num_kv_heads × head_dim")
print("4. Each KV head is shared across group of Q heads")
print("5. KV cache: Only store num_kv_heads (not num_heads)")
print("\nResult: (num_heads / num_kv_heads)× smaller KV cache!")

In [ ]:
# Test GQA
torch.manual_seed(42)

d_model = 512
num_heads = 8
num_kv_heads = 2  # 4 query heads per KV head
seq_len = 64
batch_size = 2

x = torch.randn(batch_size, seq_len, d_model, device=device)

# GQA
gqa = GroupedQueryAttention(d_model, num_heads, num_kv_heads).to(device)
gqa_output, gqa_kv = gqa(x, use_kv_cache=True)

print("Grouped-Query Attention Test")
print("=" * 70)
print(f"Configuration:")
print(f"  Num query heads: {num_heads}")
print(f"  Num KV heads: {num_kv_heads}")
print(f"  Queries per KV: {num_heads // num_kv_heads}")
print(f"\nOutput shape: {gqa_output.shape}")
print(f"KV cache K shape: {gqa_kv[0].shape}")
print(f"KV cache V shape: {gqa_kv[1].shape}")

# Compare all three
print("\n" + "=" * 70)
print("\nKV Cache Size Comparison (for this example):")
print(f"  MHA ({num_heads} KV heads):  {mha_kv_size:,} elements")
print(f"  GQA ({num_kv_heads} KV heads):  {(gqa_kv[0].numel() + gqa_kv[1].numel()):,} elements")
print(f"  MQA (1 KV head):   {mqa_kv_size:,} elements")

gqa_reduction = mha_kv_size / (gqa_kv[0].numel() + gqa_kv[1].numel())
print(f"\nGQA reduction vs MHA: {gqa_reduction:.1f}x")

## 4. Comprehensive Comparison

### MHA vs GQA vs MQA

In [ ]:
# Comprehensive comparison
print("MHA vs GQA vs MQA - Complete Comparison")
print("=" * 100)

# Llama 2 70B configuration
seq_len = 4096
num_heads = 64
head_dim = 128
num_layers = 80

# Different GQA configurations
gqa_configs = [
    ('MHA (Standard)', num_heads),
    ('GQA-32 (2 heads/group)', 32),
    ('GQA-16 (4 heads/group)', 16),
    ('GQA-8 (8 heads/group)', 8),
    ('GQA-4 (16 heads/group)', 4),
    ('MQA (All shared)', 1),
]

print(f"{'Configuration':<30} {'KV Heads':<12} {'KV Cache (GB)':<15} {'Reduction':<12} {'Quality'}")
print("-" * 100)

mha_size = None
results = []

for name, num_kv_heads in gqa_configs:
    result = calculate_kv_cache_size(seq_len, num_heads, head_dim, num_layers,
                                     num_kv_heads=num_kv_heads)
    
    if mha_size is None:
        mha_size = result['total_gb']
        reduction = 1.0
        quality = "Baseline"
    else:
        reduction = mha_size / result['total_gb']
        # Approximate quality impact (based on literature)
        if num_kv_heads == num_heads:
            quality = "Baseline"
        elif num_kv_heads >= 8:
            quality = "-0.5% ✓"
        elif num_kv_heads >= 4:
            quality = "-1.0% ✓"
        elif num_kv_heads > 1:
            quality = "-1.5%"
        else:
            quality = "-2.0%"
    
    results.append({
        'name': name,
        'num_kv_heads': num_kv_heads,
        'kv_cache_gb': result['total_gb'],
        'reduction': reduction
    })
    
    print(f"{name:<30} {num_kv_heads:<12} {result['total_gb']:<15.2f} {reduction:<12.1f}x {quality}")

print("\n" + "=" * 100)
print("\nRecommendations:")
print("• Large models (70B+): GQA-8 (8 KV heads) - Best balance")
print("• Medium models (7-13B): GQA-4 to GQA-8")
print("• Memory critical: MQA (1 KV head) - Accept quality trade-off")
print("• Quality critical: MHA (full heads) - If memory permits")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KV cache size
ax = axes[0]
names = [r['name'] for r in results]
kv_sizes = [r['kv_cache_gb'] for r in results]
colors_bar = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71', '#3498db', '#9b59b6']

bars = ax.barh(range(len(names)), kv_sizes, color=colors_bar, alpha=0.7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('KV Cache Size (GB)', fontsize=12)
ax.set_title('KV Cache Memory (Llama 2 70B, 4K ctx)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, val in zip(bars, kv_sizes):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} GB', ha='left', va='center', fontsize=9, fontweight='bold')

# Reduction factors
ax = axes[1]
reductions = [r['reduction'] for r in results]
bars = ax.bar(range(len(names)), reductions, color=colors_bar, alpha=0.7)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([r['name'].split()[0] for r in results], rotation=45, ha='right')
ax.set_ylabel('Memory Reduction Factor', fontsize=12)
ax.set_title('Memory Savings vs MHA', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, reductions):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height,
            f'{val:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. GQA-8 achieves 8x memory reduction with minimal quality loss")
print("2. MQA achieves 64x reduction but with ~2% quality degradation")
print("3. GQA is the sweet spot for most production use cases")

## 5. Real-World Model Examples

In [ ]:
print("Real-World Model Configurations")
print("=" * 100)

models = [
    {
        'name': 'Llama 2 7B',
        'attention': 'MHA (Standard)',
        'num_heads': 32,
        'num_kv_heads': 32,
        'kv_cache_4k': '1.3 GB',
        'reason': 'Smaller model, memory not critical'
    },
    {
        'name': 'Llama 2 13B',
        'attention': 'MHA (Standard)',
        'num_heads': 40,
        'num_kv_heads': 40,
        'kv_cache_4k': '2.0 GB',
        'reason': 'Smaller model, memory not critical'
    },
    {
        'name': 'Llama 2 70B',
        'attention': 'GQA',
        'num_heads': 64,
        'num_kv_heads': 8,
        'kv_cache_4k': '1.3 GB',
        'reason': 'Large model, memory critical'
    },
    {
        'name': 'Mistral 7B',
        'attention': 'GQA',
        'num_heads': 32,
        'num_kv_heads': 8,
        'kv_cache_4k': '0.3 GB',
        'reason': 'Optimized for efficiency'
    },
    {
        'name': 'Falcon 40B',
        'attention': 'MQA',
        'num_heads': 64,
        'num_kv_heads': 1,
        'kv_cache_4k': '~0.5 GB',
        'reason': 'Maximum memory efficiency'
    },
    {
        'name': 'GPT-3',
        'attention': 'MHA (Standard)',
        'num_heads': 96,
        'num_kv_heads': 96,
        'kv_cache_4k': '~12 GB',
        'reason': 'Older architecture, before MQA/GQA'
    },
]

print(f"{'Model':<18} {'Attention':<18} {'Q Heads':<10} {'KV Heads':<10} {'KV Cache':<12} {'Reason'}")
print("-" * 100)

for model in models:
    print(f"{model['name']:<18} {model['attention']:<18} {model['num_heads']:<10} "
          f"{model['num_kv_heads']:<10} {model['kv_cache_4k']:<12} {model['reason']}")

print("\n" + "=" * 100)
print("\nTrends:")
print("• Older models (GPT-3): Standard MHA")
print("• Small models (7-13B): MHA (memory not limiting)")
print("• Large models (70B+): GQA (balance quality/memory)")
print("• Efficiency-focused: MQA (maximum savings)")
print("\nFuture: GQA becoming standard for all large models")

## 6. Integration with Hugging Face Transformers

In [ ]:
print("Using GQA Models in Practice")
print("=" * 70)

usage_guide = """
1. LOADING GQA MODELS (Hugging Face):

from transformers import AutoModelForCausalLM

# Llama 2 70B (uses GQA automatically)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-70b-hf",
    torch_dtype=torch.float16,
    device_map="auto",
)

# GQA is used automatically based on model config
# No special flags needed!


2. CHECKING MODEL CONFIGURATION:

from transformers import AutoConfig

config = AutoConfig.from_pretrained("meta-llama/Llama-2-70b-hf")
print(f"Num attention heads: {config.num_attention_heads}")
print(f"Num KV heads: {config.num_key_value_heads}")

# Llama 2 70B: 64 attention heads, 8 KV heads (GQA-8)


3. CUSTOM GQA IMPLEMENTATION:

from transformers import LlamaConfig, LlamaForCausalLM

# Create model with custom GQA configuration
config = LlamaConfig(
    hidden_size=4096,
    num_attention_heads=32,
    num_key_value_heads=8,  # GQA: 4 query heads per KV head
    num_hidden_layers=32,
)

model = LlamaForCausalLM(config)


4. SERVING WITH VLLM:

from vllm import LLM

# vLLM automatically detects and uses GQA
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    dtype="float16",
    max_model_len=4096,
)

# GQA reduces KV cache → allows larger batch sizes!


5. MONITORING KV CACHE USAGE:

import torch

# Check memory before and after generation
torch.cuda.reset_peak_memory_stats()
outputs = model.generate(input_ids, max_new_tokens=100, use_cache=True)
peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak memory: {peak_mem:.2f} GB")

# With GQA: significantly lower memory usage
"""

print(usage_guide)

## 7. Decision Guide: When to Use Each

In [ ]:
print("MHA vs GQA vs MQA - Decision Guide")
print("=" * 90)

decision_guide = """
USE MULTI-HEAD ATTENTION (MHA):
✓ Small models (<13B parameters)
✓ Short contexts (<2K tokens)
✓ Quality is paramount
✓ Memory is not a constraint
✓ Small batch sizes (1-4)

Examples:
- Llama 2 7B: 32 heads, MHA
- GPT-2: 12 heads, MHA
- Research/academic models


USE GROUPED-QUERY ATTENTION (GQA):
✓ Large models (13B+ parameters) ⭐ RECOMMENDED
✓ Long contexts (4K-32K tokens)
✓ Need balance of quality and efficiency
✓ Production serving
✓ Moderate to large batch sizes

Recommended configurations:
- 7-13B models: GQA-4 to GQA-8 (4-8 KV heads)
- 30-70B models: GQA-8 (8 KV heads)
- 70B+ models: GQA-8 to GQA-16

Examples:
- Llama 2 70B: 64 heads → 8 KV heads (GQA-8)
- Mistral 7B: 32 heads → 8 KV heads (GQA-8)


USE MULTI-QUERY ATTENTION (MQA):
✓ Extreme memory constraints
✓ Very long contexts (32K+ tokens)
✓ Maximum batch size needed
✓ Can accept 1-2% quality loss
✓ Edge deployment

Examples:
- Falcon 40B: 64 heads → 1 KV head (MQA)
- Memory-optimized variants
- Mobile/edge deployments


QUICK DECISION TREE:

Model size < 13B?
├─ YES → Use MHA (memory not limiting)
└─ NO → Continue

Need best quality?
├─ YES → Use MHA or GQA-16+
└─ NO → Continue

Memory very constrained?
├─ YES → Use MQA
└─ NO → Use GQA-8 ⭐ (sweet spot)


TRAINING NEW MODELS:

1. Start with MHA for baseline
2. Try GQA with different num_kv_heads
3. Benchmark quality on your tasks
4. Choose GQA configuration with acceptable quality loss
5. Common sweet spots: GQA-4, GQA-8, GQA-16
"""

print(decision_guide)

## Summary

### Key Takeaways

1. **The KV Cache Problem**:
   - Standard MHA: KV cache scales with num_heads
   - Llama 2 70B: 10.7 GB KV cache per sequence (4K context)
   - Bottleneck for long contexts and large batches

2. **Multi-Query Attention (MQA)**:
   - Share single K, V across all query heads
   - Reduction: num_heads× smaller KV cache (8-64x)
   - Trade-off: ~1-2% quality degradation
   - Use when: Extreme memory constraints

3. **Grouped-Query Attention (GQA)**:
   - Middle ground: Group query heads, share K,V within groups
   - Reduction: (num_heads / num_kv_heads)× smaller
   - Trade-off: <1% quality degradation (minimal!)
   - Sweet spot: GQA-8 (8 KV heads) for most large models

4. **Real-World Adoption**:
   - Llama 2 70B: GQA with 8 KV heads (64 query heads)
   - Mistral 7B: GQA with 8 KV heads (32 query heads)
   - Falcon: MQA with 1 KV head
   - Trend: GQA becoming standard for large models

5. **Memory Savings** (Llama 2 70B, 4K context):
   - MHA: 10.7 GB
   - GQA-8: 1.3 GB (8x reduction)
   - MQA: 0.17 GB (64x reduction)

6. **Quality vs Efficiency**:
   - MHA: Best quality, largest memory
   - GQA: Excellent trade-off (recommended)
   - MQA: Maximum efficiency, acceptable quality

### Implementation Notes

**Key Differences**:
```python
# MHA
K: [batch, seq, num_heads, head_dim]
V: [batch, seq, num_heads, head_dim]

# GQA  
K: [batch, seq, num_kv_heads, head_dim]
V: [batch, seq, num_kv_heads, head_dim]
# Then repeat_interleave to match num_heads

# MQA
K: [batch, seq, 1, head_dim]
V: [batch, seq, 1, head_dim]
# Then expand to match num_heads
```

### Historical Impact

- **2019**: MQA introduced (Shazeer, "Fast Transformer Decoding")
- **2020-2022**: Limited adoption (quality concerns)
- **2023**: GQA paper (Ainslie et al.) - Best of both worlds
- **2023**: Llama 2 70B uses GQA - Validation in production
- **2024**: GQA standard for new large models

### Next Steps

- **Notebook 20**: Flash Attention (attention computation optimization)
- **Notebook 23**: Long Context Inference (32K+ tokens with GQA)
- **Notebook 24**: Memory Optimization Comparison (all techniques)
- **Notebook 55**: KV Cache Optimization (quantization, compression)

### Further Reading

- Paper: "Fast Transformer Decoding: One Write-Head is All You Need" (Shazeer, 2019)
- Paper: "GQA: Training Generalized Multi-Query Transformer" (Ainslie et al., 2023)
- Llama 2 Paper: Details on GQA implementation
- LLM_INFERENCE_ROADMAP.md - Stage 2: Memory Optimization